# 06 · LSTM & GRU untuk Deret Waktu — Bab 7

*Pengantar Deep Learning untuk Meteorologi* · Kanada Kurniawan

Notebook pendamping Bab 7: membangun LSTM/GRU untuk prediksi deret waktu, membandingkan dengan *baseline* (persistence/klimatologi), dan mengevaluasi dengan walk-forward. Menggunakan data sintetik pasang surut agar mudah dijalankan.

## 1. Setup & Data

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow:", tf.__version__)

# data pasang surut sintetik (periodik + noise)
t = np.arange(0, 3000)
ts = np.sin(2*np.pi*t/24) * 0.6 + np.sin(2*np.pi*t/12.42) * 0.4 + 0.05*np.random.randn(len(t))
ts = ts + 1.0  # baseline tinggi
print("n =", len(ts))

## 2. Windowing (Kode 7.1)

In [ ]:
def buat_window(deret, w=24, h=1):
    X, y = [], []
    for i in range(len(deret) - w - h + 1):
        X.append(deret[i:i+w])
        y.append(deret[i+w:i+w+h])
    return np.array(X), np.array(y)

w, h = 24, 1
X, y = buat_window(ts, w=w, h=h)
print("X shape:", X.shape, "| y shape:", y.shape)

# split berbasis waktu
n = len(X)
ntr, nva = int(n*0.7), int(n*0.15)
Xtr, ytr = X[:ntr], y[:ntr]
Xva, yva = X[ntr:ntr+nva], y[ntr:ntr+nva]
Xte, yte = X[ntr+nva:], y[ntr+nva:]
print("train", Xtr.shape, "val", Xva.shape, "test", Xte.shape)

## 3. Baseline Dulu (Kode 7.2)

In [ ]:
def mae(a, b):
    return float(np.mean(np.abs(a - b)))

# persistence: pakai nilai terakhir tiap window
base_persist = Xte[:, -1, 0]
# klimatologi: rata-rata train
base_klimat = np.full(len(yte), float(np.mean(ytr)))

print("MAE persistence:", round(mae(yte.ravel(), base_persist.ravel()), 4))
print("MAE klimatologi:", round(mae(yte.ravel(), base_klimat.ravel()), 4))

## 4. Model LSTM (Kode 7.3) & GRU

Bandingkan LSTM satu lapisan vs GRU satu lapisan.

In [ ]:
def build_seq(kind="lstm", units=16):
    layer = tf.keras.layers.LSTM if kind == "lstm" else tf.keras.layers.GRU
    model = tf.keras.Sequential([
        layer(units, input_shape=(w, 1)),
        tf.keras.layers.Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

results = {}
for kind in ["lstm", "gru"]:
    m = build_seq(kind)
    m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=30, batch_size=32, verbose=0)
    p = m.predict(Xte, verbose=0).ravel()
    results[kind] = mae(yte.ravel(), p)
    print(f"MAE {kind.upper()}: {results[kind]:.4f}")

print("MAE persistence:", round(mae(yte.ravel(), base_persist.ravel()), 4))

## 5. Multivariate LSTM (Kode 7.4)

Tambahkan satu fitur (misal suhu sintetik) sehingga input berisi 2 fitur.

In [ ]:
suhu = 27 + 0.5*np.cos(2*np.pi*t/720) + 0.3*np.random.randn(len(t))
# susun: [pasang, suhu] per langkah waktu
data2 = np.stack([ts, suhu], axis=1)

def buat_window_2f(deret2d, w=24, h=1):
    X, y = [], []
    for i in range(len(deret2d) - w - h + 1):
        X.append(deret2d[i:i+w])
        y.append(deret2d[i+w:i+w+h, 0])  # target tetap fitur pertama (pasang)
    return np.array(X), np.array(y)

X2, y2 = buat_window_2f(data2, w=w, h=h)

n2 = len(X2)
ntr2, nva2 = int(n2*0.7), int(n2*0.15)
X2tr, y2tr = X2[:ntr2], y2[:ntr2]
X2va, y2va = X2[ntr2:ntr2+nva2], y2[ntr2:ntr2+nva2]
X2te, y2te = X2[ntr2+nva2:], y2[ntr2+nva2:]

m2 = tf.keras.Sequential([
    tf.keras.layers.LSTM(16, input_shape=(w, 2)),
    tf.keras.layers.Dense(1),
])
m2.compile(optimizer="adam", loss="mse", metrics=["mae"])
m2.fit(X2tr, y2tr, validation_data=(X2va, y2va), epochs=30, batch_size=32, verbose=0)
p2 = m2.predict(X2te, verbose=0).ravel()
print("MAE LSTM multivariate:", round(mae(y2te.ravel(), p2), 4))

## 6. Plot Forecast vs Aktual (Kode 7.5)

Latih ulang LSTM singkat dan simpan prediksi untuk diplot (agar sel berdiri sendiri).

In [ ]:
mplot = build_seq("lstm", units=16)
mplot.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=20, batch_size=32, verbose=0)
pred_lstm = mplot.predict(Xte, verbose=0).ravel()

plt.figure(figsize=(11, 3.2))
plt.plot(yte[:300].ravel(), label="aktual", lw=1.4)
plt.plot(pred_lstm[:300], label="prediksi LSTM", lw=1.1, alpha=0.85)
plt.legend(); plt.tight_layout(); plt.show()
print("MAE LSTM (plot):", round(mae(yte.ravel(), pred_lstm), 4))

## 7. Latihan Mini

1. Ubah `w` ∈ {6, 12, 24, 48} dan catat MAE tiap konfigurasi.
2. Ganti `units` dan jumlah lapisan (`return_sequences=True` + LSTM kedua) — bandingkan kurva val loss.
3. Terapkan `Dropout(0.2)` dan `L2` pada LSTM — panel banding.
4. Untuk h=1..7, bangun model *direct* per lead time; plot MAE per horizon (Kode/§7.8).
5. Hitung skill score `SS = 1 - MAE_model/MAE_persistence`.